# Advanced Problems with Solutions: Selecting and Filtering Iterators

Topic focus: `filter`, `itertools.filterfalse`, `itertools.takewhile`, `itertools.dropwhile`, and `itertools.compress`.

This notebook is a practice-heavy extension of the selecting/filtering iterators lesson. It emphasizes:

- lazy evaluation
- single-pass iterator consumption
- predicate design
- truthiness
- filtering finite and infinite iterables
- prefix/suffix selection with `takewhile` and `dropwhile`
- mask-based selection with `compress`
- robust streaming pipelines
- common gotchas and best practices

Each problem includes a complete solution and runnable checks.

## Best-practice checklist

1. Prefer lazy iterator tools when data may be large or infinite.
2. Keep predicates small, named, and testable.
3. Avoid side effects inside predicates unless you are intentionally tracing/debugging.
4. Remember that `filter`, `filterfalse`, `takewhile`, `dropwhile`, and `compress` return iterators.
5. Remember that iterators are consumed. Reusing the same filtering iterator usually gives no results the second time.
6. Use `filterfalse(pred, data)` instead of `filter(lambda x: not pred(x), data)` when it improves clarity.
7. Use `takewhile` only for prefix logic. It stops at the first failed predicate.
8. Use `dropwhile` only for dropping an initial prefix. After the first failed predicate, it stops checking the predicate.
9. Use `compress(data, selectors)` when a separate truth-mask controls which records are kept.
10. Be careful with `itertools.tee`: it supports multiple lazy views, but may cache many values if the views are consumed at very different speeds.

## Setup utilities

The helpers below make laziness and consumption visible.

In [1]:
from itertools import (
    filterfalse,
    takewhile,
    dropwhile,
    compress,
    islice,
    count,
    tee,
)
from math import sin, pi
from collections import deque
import re


def traced(iterable, label="item"):
    # Yield values while printing when the source is actually consumed.
    for x in iterable:
        print(f"{label} -> {x!r}")
        yield x


def take(n, iterable):
    # Return the first n values from any iterable as a list.
    return list(islice(iterable, n))


def consume(iterable):
    # Fully consume an iterable without keeping the values.
    deque(iterable, maxlen=0)


def assert_iterable_equal(actual_iterable, expected):
    actual = list(actual_iterable)
    assert actual == expected, f"Expected {expected!r}, got {actual!r}"
    return actual

---

# Problem 1 — Build reusable predicates for cube filtering

Write a generator `gen_cubes(n)` that yields the cubes of `0..n-1`.

Then use a named predicate `is_odd(x)` with `filter` to keep only odd cubes.

Expected result for `n=10`:

```python
[1, 27, 125, 343, 729]
```

In [2]:
# Solution

def gen_cubes(n):
    for i in range(n):
        yield i ** 3


def is_odd(x):
    return x % 2 == 1


odd_cubes = list(filter(is_odd, gen_cubes(10)))
odd_cubes

[1, 27, 125, 343, 729]

In [3]:
# Checks
assert odd_cubes == [1, 27, 125, 343, 729]
assert list(gen_cubes(5)) == [0, 1, 8, 27, 64]

# Problem 2 — Prove that `filter` is lazy

Create a traced cube generator and wrap it in `filter`. Confirm that no source values are produced until the filtered iterator is consumed.

In [4]:
# Solution

def gen_cubes_traced(n):
    for i in range(n):
        print(f"source generated i={i}")
        yield i ** 3


filtered = filter(is_odd, gen_cubes_traced(5))
print("filter object created; source has not run yet")

first_value = next(filtered)
print("first kept value:", first_value)

remaining_values = list(filtered)
print("remaining kept values:", remaining_values)

filter object created; source has not run yet
source generated i=0
source generated i=1
first kept value: 1
source generated i=2
source generated i=3
source generated i=4
remaining kept values: [27]


In [5]:
# Checks
assert first_value == 1
assert remaining_values == [27]

# Problem 3 — Use `filter(None, data)` carefully

`filter(None, data)` keeps truthy values and removes falsy values.

Use it to clean a mixed list, then write an explicit predicate that removes only `None` values while preserving `0`, `False`, and empty strings.

In [6]:
# Solution

data = [0, 1, "", "python", None, [], [1, 2], False, True, {}, {"x": 1}]

truthy_only = list(filter(None, data))

def is_not_none(x):
    return x is not None

without_none = list(filter(is_not_none, data))

truthy_only, without_none

([1, 'python', [1, 2], True, {'x': 1}],
 [0, 1, '', 'python', [], [1, 2], False, True, {}, {'x': 1}])

In [7]:
# Checks
assert truthy_only == [1, "python", [1, 2], True, {"x": 1}]
assert without_none == [0, 1, "", "python", [], [1, 2], False, True, {}, {"x": 1}]

# Problem 4 — Replace predicate inversion with `filterfalse`

Using only `is_odd`, keep the even cubes from `gen_cubes(10)`.

Do this once with `filterfalse` and once with a lambda-based inversion. Confirm both results match.

In [8]:
# Solution

even_cubes_a = list(filterfalse(is_odd, gen_cubes(10)))
even_cubes_b = list(filter(lambda x: not is_odd(x), gen_cubes(10)))

even_cubes_a, even_cubes_b

([0, 8, 64, 216, 512], [0, 8, 64, 216, 512])

In [9]:
# Checks
assert even_cubes_a == [0, 8, 64, 216, 512]
assert even_cubes_a == even_cubes_b

# Problem 5 — Implement your own lazy `filter`

Write `my_filter(predicate, iterable)`.

Requirements:

- it must be lazy
- it must support `predicate=None`, matching the behavior of built-in `filter(None, iterable)`
- it must not build a list internally

In [10]:
# Solution

def my_filter(predicate, iterable):
    if predicate is None:
        predicate = bool
    for item in iterable:
        if predicate(item):
            yield item


assert_iterable_equal(my_filter(is_odd, gen_cubes(10)), [1, 27, 125, 343, 729])
assert_iterable_equal(my_filter(None, [0, "", "x", [], [1], None, 10]), ["x", [1], 10])

['x', [1], 10]

# Problem 6 — Implement your own lazy `filterfalse`

Write `my_filterfalse(predicate, iterable)`.

Requirements:

- it must be lazy
- it must support `predicate=None`
- if `predicate is None`, it should yield falsy values

In [11]:
# Solution

def my_filterfalse(predicate, iterable):
    if predicate is None:
        predicate = bool
    for item in iterable:
        if not predicate(item):
            yield item


assert_iterable_equal(my_filterfalse(is_odd, gen_cubes(10)), [0, 8, 64, 216, 512])
assert_iterable_equal(my_filterfalse(None, [0, "", "x", [], [1], None, 10]), [0, "", [], None])

[0, '', [], None]

# Problem 7 — Partition an iterable into accepted and rejected values

Write `partition(predicate, iterable)` that returns two lazy iterators:

```python
rejects, accepts = partition(predicate, iterable)
```

Use `tee`, `filter`, and `filterfalse`.

Important: this is lazy, but `tee` may cache values if one returned iterator is consumed far ahead of the other.

In [12]:
# Solution

def partition(predicate, iterable):
    a, b = tee(iterable)
    return filterfalse(predicate, a), filter(predicate, b)


rejects, accepts = partition(is_odd, range(10))

reject_list = list(rejects)
accept_list = list(accepts)

reject_list, accept_list

([0, 2, 4, 6, 8], [1, 3, 5, 7, 9])

In [13]:
# Checks
assert reject_list == [0, 2, 4, 6, 8]
assert accept_list == [1, 3, 5, 7, 9]

# Problem 8 — Single-pass eager partition

Sometimes you want to avoid `tee` caching. Write `partition_lists(predicate, iterable)` that consumes the iterable once and returns two lists.

This is not lazy, but it uses predictable memory and avoids hidden `tee` cache growth.

In [14]:
# Solution

def partition_lists(predicate, iterable):
    rejects = []
    accepts = []
    for item in iterable:
        if predicate(item):
            accepts.append(item)
        else:
            rejects.append(item)
    return rejects, accepts


rejects, accepts = partition_lists(lambda x: x >= 5, range(10))
rejects, accepts

([0, 1, 2, 3, 4], [5, 6, 7, 8, 9])

In [15]:
# Checks
assert rejects == [0, 1, 2, 3, 4]
assert accepts == [5, 6, 7, 8, 9]

# Problem 9 — Filter an infinite stream

Use `itertools.count` to create an infinite stream of integers. Keep numbers divisible by 7 but not by 5. Return the first 12 matching values.

Do not materialize the infinite stream.

In [16]:
# Solution

def divisible_by_7_not_5(x):
    return x % 7 == 0 and x % 5 != 0


first_12 = take(12, filter(divisible_by_7_not_5, count(1)))
first_12

[7, 14, 21, 28, 42, 49, 56, 63, 77, 84, 91, 98]

In [17]:
# Checks
assert first_12 == [7, 14, 21, 28, 42, 49, 56, 63, 77, 84, 91, 98]

# Problem 10 — Prime stream with `filter`

Build a simple predicate `is_prime(n)`, then use it with `filter` and `count` to get the first 20 primes.

This is not the fastest prime generator; the goal is clean predicate design and lazy selection.

In [18]:
# Solution

def is_prime(n):
    if n < 2:
        return False
    if n == 2:
        return True
    if n % 2 == 0:
        return False
    limit = int(n ** 0.5)
    for divisor in range(3, limit + 1, 2):
        if n % divisor == 0:
            return False
    return True


first_20_primes = take(20, filter(is_prime, count(2)))
first_20_primes

[2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71]

In [19]:
# Checks
assert first_20_primes == [2, 3, 5, 7, 11, 13, 17, 19, 23, 29,
                           31, 37, 41, 43, 47, 53, 59, 61, 67, 71]

# Problem 11 — `takewhile` is not a filter

Create a sine wave with 15 rounded values. Compare:

```python
takewhile(lambda x: 0 <= x <= 0.9, sine_wave(15))
filter(lambda x: 0 <= x <= 0.9, sine_wave(15))
```

Explain the difference in the result.

In [20]:
# Solution

def sine_wave(n):
    start = 0
    stop = 2 * pi
    step = (stop - start) / (n - 1)
    for _ in range(n):
        yield round(sin(start), 2)
        start += step


wave = list(sine_wave(15))

prefix_values = list(takewhile(lambda x: 0 <= x <= 0.9, sine_wave(15)))
filtered_values = list(filter(lambda x: 0 <= x <= 0.9, sine_wave(15)))

wave, prefix_values, filtered_values

([0.0,
  0.43,
  0.78,
  0.97,
  0.97,
  0.78,
  0.43,
  0.0,
  -0.43,
  -0.78,
  -0.97,
  -0.97,
  -0.78,
  -0.43,
  -0.0],
 [0.0, 0.43, 0.78],
 [0.0, 0.43, 0.78, 0.78, 0.43, 0.0, -0.0])

In [21]:
# Checks
assert prefix_values == [0.0, 0.43, 0.78]
assert filtered_values == [0.0, 0.43, 0.78, 0.78, 0.43, 0.0, -0.0]

`takewhile` keeps only the initial prefix while the predicate remains true. It stops forever at the first failure.

`filter` checks every value and keeps all values where the predicate is true.

# Problem 12 — Implement your own lazy `takewhile`

Write `my_takewhile(predicate, iterable)`.

It should stop immediately when the predicate first returns false.

In [22]:
# Solution

def my_takewhile(predicate, iterable):
    for item in iterable:
        if predicate(item):
            yield item
        else:
            break


assert_iterable_equal(my_takewhile(lambda x: x < 5, [1, 3, 4, 5, 1, 2]), [1, 3, 4])
assert_iterable_equal(my_takewhile(lambda x: x >= 0, [0, 2, -1, 3, 4]), [0, 2])

[0, 2]

# Problem 13 — Implement your own lazy `dropwhile`

Write `my_dropwhile(predicate, iterable)`.

It should drop initial values while the predicate is true. Once the predicate first becomes false, it should yield that item and every later item without checking the predicate again.

In [23]:
# Solution

def my_dropwhile(predicate, iterable):
    iterator = iter(iterable)

    for item in iterator:
        if not predicate(item):
            yield item
            break

    for item in iterator:
        yield item


assert_iterable_equal(my_dropwhile(lambda x: x < 5, [1, 3, 5, 2, 1]), [5, 2, 1])
assert_iterable_equal(my_dropwhile(lambda x: x != "START", ["skip", "skip", "START", "a", "b"]), ["START", "a", "b"])

['START', 'a', 'b']

# Problem 14 — Count predicate calls for `takewhile` and `dropwhile`

Use a stateful predicate counter to show:

- `takewhile` calls the predicate on the first failing item, but does not yield that item.
- `dropwhile` calls the predicate until the first failing item, yields that item, and then stops calling the predicate for all later items.

In [24]:
# Solution

class PredicateCounter:
    def __init__(self, predicate):
        self.predicate = predicate
        self.calls = []

    def __call__(self, x):
        self.calls.append(x)
        return self.predicate(x)


p1 = PredicateCounter(lambda x: x < 4)
taken = list(takewhile(p1, [1, 2, 3, 4, 1, 2]))

p2 = PredicateCounter(lambda x: x < 4)
dropped = list(dropwhile(p2, [1, 2, 3, 4, 1, 2]))

taken, p1.calls, dropped, p2.calls

([1, 2, 3], [1, 2, 3, 4], [4, 1, 2], [1, 2, 3, 4])

In [25]:
# Checks
assert taken == [1, 2, 3]
assert p1.calls == [1, 2, 3, 4]

assert dropped == [4, 1, 2]
assert p2.calls == [1, 2, 3, 4]

# Problem 15 — Parse a log stream using `dropwhile` and `takewhile`

Given a stream of log lines:

1. Drop everything before `"BEGIN DATA"`.
2. Drop the marker itself.
3. Take everything until `"END DATA"` appears.
4. Parse integer values from lines like `"value=10"`.

Do not use list slicing.

In [26]:
# Solution

log_lines = iter([
    "metadata: run=42",
    "metadata: user=ada",
    "BEGIN DATA",
    "value=10",
    "value=15",
    "value=21",
    "END DATA",
    "footer: ignored",
])

after_marker = dropwhile(lambda line: line != "BEGIN DATA", log_lines)
next(after_marker)  # consume the marker itself

data_lines = takewhile(lambda line: line != "END DATA", after_marker)
values = [int(line.split("=")[1]) for line in data_lines]

values

[10, 15, 21]

In [27]:
# Checks
assert values == [10, 15, 21]

# Problem 16 — Keep a suffix after a warm-up period

You receive sensor readings. Drop readings while they are below `70`. Once a reading is at least `70`, keep every later reading, even if some later readings drop below `70`.

Use `dropwhile`.

In [28]:
# Solution

readings = [62, 65, 69, 70, 68, 75, 71, 66]

after_warmup = list(dropwhile(lambda x: x < 70, readings))
after_warmup

[70, 68, 75, 71, 66]

In [29]:
# Checks
assert after_warmup == [70, 68, 75, 71, 66]

# Problem 17 — Stop a stream at a sentinel

Given a stream of messages, keep messages until the first `"STOP"` sentinel.

Use `takewhile`.

In [30]:
# Solution

messages = iter(["alpha", "beta", "gamma", "STOP", "delta", "epsilon"])

before_stop = list(takewhile(lambda msg: msg != "STOP", messages))
before_stop

['alpha', 'beta', 'gamma']

In [31]:
# Checks
assert before_stop == ["alpha", "beta", "gamma"]
# The STOP value was consumed by takewhile because it had to test it.
assert next(messages) == "delta"

# Problem 18 — Keep the sentinel using a custom helper

Standard `takewhile` does not yield the failing item. Write `take_until(predicate, iterable, include_match=True)`.

It should yield values until `predicate(item)` becomes true. If `include_match=True`, include the matching item.

In [32]:
# Solution

def take_until(predicate, iterable, *, include_match=True):
    for item in iterable:
        if predicate(item):
            if include_match:
                yield item
            break
        yield item


stream = ["alpha", "beta", "STOP", "gamma"]

with_stop = list(take_until(lambda x: x == "STOP", stream, include_match=True))
without_stop = list(take_until(lambda x: x == "STOP", stream, include_match=False))

with_stop, without_stop

(['alpha', 'beta', 'STOP'], ['alpha', 'beta'])

In [33]:
# Checks
assert with_stop == ["alpha", "beta", "STOP"]
assert without_stop == ["alpha", "beta"]

# Problem 19 — Mask-based selection with `compress`

Given data and selectors, keep values whose matching selector is truthy.

Also show an equivalent generator expression using `zip`.

In [34]:
# Solution

data = ["a", "b", "c", "d", "e"]
selectors = [True, False, 1, 0, "yes"]

selected_a = list(compress(data, selectors))
selected_b = [item for item, selector in zip(data, selectors) if selector]

selected_a, selected_b

(['a', 'c', 'e'], ['a', 'c', 'e'])

In [35]:
# Checks
assert selected_a == ["a", "c", "e"]
assert selected_a == selected_b

# Problem 20 — `compress` stops at the shorter iterable

Demonstrate that `compress(data, selectors)` stops when either `data` or `selectors` is exhausted.

Create one example with fewer selectors than data, and one with fewer data items than selectors.

In [36]:
# Solution

more_data_than_selectors = list(compress(["a", "b", "c", "d"], [1, 0]))
more_selectors_than_data = list(compress(["a", "b"], [0, 1, 1, 1, 1]))

more_data_than_selectors, more_selectors_than_data

(['a'], ['b'])

In [37]:
# Checks
assert more_data_than_selectors == ["a"]
assert more_selectors_than_data == ["b"]

# Problem 21 — Build selectors from a second stream

You have records and quality scores. Keep records with a score of at least `0.8`.

Use `compress`, where the selector stream is computed lazily from the score stream.

In [38]:
# Solution

records = [
    {"id": "A", "value": 10},
    {"id": "B", "value": 20},
    {"id": "C", "value": 30},
    {"id": "D", "value": 40},
]

scores = [0.91, 0.40, 0.85, 0.79]

selectors = (score >= 0.8 for score in scores)
high_quality_records = list(compress(records, selectors))

high_quality_records

[{'id': 'A', 'value': 10}, {'id': 'C', 'value': 30}]

In [39]:
# Checks
assert high_quality_records == [
    {"id": "A", "value": 10},
    {"id": "C", "value": 30},
]

# Problem 22 — Combine `filter` and `compress`

You receive transaction records. Keep only approved transactions, then use a selector mask to audit only some of those approved transactions.

Important: the mask applies after filtering.

In [40]:
# Solution

transactions = [
    {"id": 1, "status": "approved", "amount": 100},
    {"id": 2, "status": "rejected", "amount": 200},
    {"id": 3, "status": "approved", "amount": 300},
    {"id": 4, "status": "approved", "amount": 400},
    {"id": 5, "status": "rejected", "amount": 500},
    {"id": 6, "status": "approved", "amount": 600},
]

def is_approved(tx):
    return tx["status"] == "approved"

approved = filter(is_approved, transactions)
audit_mask_for_approved_only = [True, False, True, False]

audit_sample = list(compress(approved, audit_mask_for_approved_only))

audit_sample

[{'id': 1, 'status': 'approved', 'amount': 100},
 {'id': 4, 'status': 'approved', 'amount': 400}]

In [41]:
# Checks
assert [tx["id"] for tx in audit_sample] == [1, 4]

# Problem 23 — Create predicate factories

Write predicate factory functions:

- `field_equals(field, expected)`
- `field_between(field, low, high, inclusive=True)`
- `all_of(*predicates)`
- `any_of(*predicates)`
- `not_(predicate)`

Use them to filter a list of dictionaries.

In [42]:
# Solution

def field_equals(field, expected):
    return lambda record: record.get(field) == expected


def field_between(field, low, high, *, inclusive=True):
    if inclusive:
        return lambda record: low <= record.get(field) <= high
    return lambda record: low < record.get(field) < high


def all_of(*predicates):
    return lambda x: all(predicate(x) for predicate in predicates)


def any_of(*predicates):
    return lambda x: any(predicate(x) for predicate in predicates)


def not_(predicate):
    return lambda x: not predicate(x)


people = [
    {"name": "Ada", "team": "red", "score": 91, "active": True},
    {"name": "Ben", "team": "blue", "score": 88, "active": False},
    {"name": "Cal", "team": "red", "score": 72, "active": True},
    {"name": "Dee", "team": "blue", "score": 95, "active": True},
    {"name": "Eli", "team": "red", "score": 99, "active": False},
]

red = field_equals("team", "red")
active = field_equals("active", True)
score_80_100 = field_between("score", 80, 100)

selected_people = list(filter(all_of(red, active, score_80_100), people))
selected_people

[{'name': 'Ada', 'team': 'red', 'score': 91, 'active': True}]

In [43]:
# Checks
assert selected_people == [{"name": "Ada", "team": "red", "score": 91, "active": True}]

inactive_or_blue = list(filter(any_of(not_(active), field_equals("team", "blue")), people))
assert [person["name"] for person in inactive_or_blue] == ["Ben", "Dee", "Eli"]

# Problem 24 — Streaming validation pipeline

You have raw records with messy values. Build a lazy pipeline that:

1. drops records before the first record with `kind == "data"`
2. keeps only records with `kind == "data"`
3. rejects records whose numeric value cannot be parsed
4. keeps values in the range `[10, 50]`
5. stops at the first record with `kind == "end"`

Return the cleaned numeric values.

In [44]:
# Solution

raw_records = iter([
    {"kind": "meta", "value": None},
    {"kind": "meta", "value": "ignored"},
    {"kind": "data", "value": "8"},
    {"kind": "data", "value": "10"},
    {"kind": "data", "value": "bad"},
    {"kind": "data", "value": "35"},
    {"kind": "data", "value": "60"},
    {"kind": "end", "value": None},
    {"kind": "data", "value": "40"},  # ignored after end
])

def is_data(record):
    return record["kind"] == "data"

def is_end(record):
    return record["kind"] == "end"

def parse_int_or_none(record):
    try:
        return int(record["value"])
    except (TypeError, ValueError):
        return None

after_first_data = dropwhile(lambda r: not is_data(r), raw_records)
before_end = takewhile(lambda r: not is_end(r), after_first_data)
data_only = filter(is_data, before_end)
parsed_values = map(parse_int_or_none, data_only)
valid_numbers = filter(lambda x: x is not None, parsed_values)
in_range = filter(lambda x: 10 <= x <= 50, valid_numbers)

clean_values = list(in_range)
clean_values

[10, 35]

In [45]:
# Checks
assert clean_values == [10, 35]

# Problem 25 — Show iterator consumption after partial filtering

Create an iterator over `range(10)`, filter for even values, call `next()` once, then convert the rest to a list.

Show that the original iterator has also advanced.

In [46]:
# Solution

source = iter(range(10))
evens = filter(lambda x: x % 2 == 0, source)

first_even = next(evens)
remaining_evens = list(evens)
remaining_source = list(source)

first_even, remaining_evens, remaining_source

(0, [2, 4, 6, 8], [])

In [47]:
# Checks
assert first_even == 0
assert remaining_evens == [2, 4, 6, 8]
assert remaining_source == []

Why is `remaining_source` empty?

The `filter` iterator had to consume the source until it was exhausted while finding all remaining even values. It discarded the odd values internally, but those odd values were still consumed from `source`.

# Problem 26 — Limit the work done by a filter over an expensive stream

You have an expensive infinite stream. Use `filter` plus `islice` to keep the first 5 values divisible by 13.

Also count how many source items had to be generated.

In [48]:
# Solution

class CountingIntegers:
    def __init__(self, start=1):
        self.current = start
        self.generated = 0

    def __iter__(self):
        return self

    def __next__(self):
        value = self.current
        self.current += 1
        self.generated += 1
        return value


stream = CountingIntegers(1)
first_five_multiples_of_13 = list(islice(filter(lambda x: x % 13 == 0, stream), 5))

first_five_multiples_of_13, stream.generated

([13, 26, 39, 52, 65], 65)

In [49]:
# Checks
assert first_five_multiples_of_13 == [13, 26, 39, 52, 65]
assert stream.generated == 65

Filtering is lazy, but laziness does not mean free. The upstream iterator still has to produce every skipped value needed to find the kept values.

# Problem 27 — Replace eager list comprehensions with lazy pipelines

Convert this eager expression into a lazy pipeline:

```python
[x * 10 for x in numbers if x % 3 == 0 if x > 10]
```

Return only the first 4 values from the lazy pipeline.

In [50]:
# Solution

numbers = count(1)

pipeline = map(
    lambda x: x * 10,
    filter(lambda x: x > 10, filter(lambda x: x % 3 == 0, numbers))
)

first_four = take(4, pipeline)
first_four

[120, 150, 180, 210]

In [51]:
# Checks
assert first_four == [120, 150, 180, 210]

A generator expression is often cleaner for simple cases:

```python
(x * 10 for x in count(1) if x % 3 == 0 if x > 10)
```

The iterator tools are useful when you want named steps, composability, or reusable functions.

In [52]:
# Equivalent generator-expression solution

numbers = count(1)
pipeline_genexpr = (x * 10 for x in numbers if x % 3 == 0 if x > 10)

assert take(4, pipeline_genexpr) == [120, 150, 180, 210]

# Problem 28 — Use `dropwhile` and `takewhile` for time-window selection

Given events sorted by timestamp, select only events in the half-open interval:

```python
start <= timestamp < stop
```

Use `dropwhile` to skip events before the start and `takewhile` to stop before the end.

In [53]:
# Solution

events = [
    {"t": 1, "name": "boot"},
    {"t": 3, "name": "load_config"},
    {"t": 5, "name": "ready"},
    {"t": 8, "name": "request_a"},
    {"t": 9, "name": "request_b"},
    {"t": 12, "name": "shutdown"},
]

start = 5
stop = 10

window = takewhile(
    lambda event: event["t"] < stop,
    dropwhile(lambda event: event["t"] < start, events)
)

window_events = list(window)
window_events

[{'t': 5, 'name': 'ready'},
 {'t': 8, 'name': 'request_a'},
 {'t': 9, 'name': 'request_b'}]

In [54]:
# Checks
assert [event["name"] for event in window_events] == ["ready", "request_a", "request_b"]

This works correctly because the events are sorted. If events were unsorted, `filter` would be the correct tool for selecting all matching events.

# Problem 29 — Compare sorted-stream windowing with unsorted filtering

Use the same time interval as Problem 28, but with unsorted events.

Show that `dropwhile`/`takewhile` gives the wrong answer, while `filter` gives all matching records.

In [55]:
# Solution

unsorted_events = [
    {"t": 8, "name": "request_a"},
    {"t": 1, "name": "boot"},
    {"t": 12, "name": "shutdown"},
    {"t": 5, "name": "ready"},
    {"t": 9, "name": "request_b"},
]

wrong_window = list(takewhile(
    lambda event: event["t"] < stop,
    dropwhile(lambda event: event["t"] < start, unsorted_events)
))

correct_window = list(filter(lambda event: start <= event["t"] < stop, unsorted_events))

wrong_window, correct_window

([{'t': 8, 'name': 'request_a'}, {'t': 1, 'name': 'boot'}],
 [{'t': 8, 'name': 'request_a'},
  {'t': 5, 'name': 'ready'},
  {'t': 9, 'name': 'request_b'}])

In [56]:
# Checks
assert [event["name"] for event in wrong_window] == ["request_a", "boot"]
assert [event["name"] for event in correct_window] == ["request_a", "ready", "request_b"]

`dropwhile` and `takewhile` are positional/prefix tools. Use them when order itself carries meaning, such as sorted time streams, headers/footers, or warm-up phases.

# Problem 30 — Build a robust line filter

Given text lines, keep only meaningful lines:

- remove leading and trailing whitespace
- ignore blank lines
- ignore comment lines starting with `#`
- stop at a line equal to `"__END__"`
- parse `key=value` pairs into tuples

Use a lazy pipeline.

In [57]:
# Solution

lines = iter([
    "   ",
    "# config generated by system",
    "host = localhost",
    "port= 5432",
    "",
    "# debug=false",
    "user = ada",
    "__END__",
    "password = should_not_be_read",
])

stripped = map(str.strip, lines)
before_end = takewhile(lambda line: line != "__END__", stripped)
not_blank = filter(None, before_end)
not_comments = filterfalse(lambda line: line.startswith("#"), not_blank)

def parse_key_value(line):
    key, value = line.split("=", 1)
    return key.strip(), value.strip()

pairs = list(map(parse_key_value, not_comments))
pairs

[('host', 'localhost'), ('port', '5432'), ('user', 'ada')]

In [58]:
# Checks
assert pairs == [("host", "localhost"), ("port", "5432"), ("user", "ada")]

# Problem 31 — Select valid emails using regex predicates

Create a predicate factory `matches(pattern)` and use it to keep email-like strings.

Then use `filterfalse` to collect invalid values.

In [59]:
# Solution

def matches(pattern):
    regex = re.compile(pattern)
    return lambda text: bool(regex.fullmatch(text))


email_like = matches(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")

candidates = [
    "ada@example.com",
    "not an email",
    "ben@company",
    "cal@sub.example.org",
    "@missing-local.com",
    "dee@example.net",
]

valid_emails = list(filter(email_like, candidates))
invalid_emails = list(filterfalse(email_like, candidates))

valid_emails, invalid_emails

(['ada@example.com', 'cal@sub.example.org', 'dee@example.net'],
 ['not an email', 'ben@company', '@missing-local.com'])

In [60]:
# Checks
assert valid_emails == ["ada@example.com", "cal@sub.example.org", "dee@example.net"]
assert invalid_emails == ["not an email", "ben@company", "@missing-local.com"]

# Problem 32 — Use `compress` for feature flags

You have feature names and enabled/disabled flags. Return enabled features.

Then create a second selector stream that enables features whose names contain `"search"`.

In [61]:
# Solution

features = ["login", "search_basic", "billing", "search_advanced", "export"]
enabled_flags = [True, True, False, False, True]

enabled_features = list(compress(features, enabled_flags))

search_flags = ("search" in feature for feature in features)
search_features = list(compress(features, search_flags))

enabled_features, search_features

(['login', 'search_basic', 'export'], ['search_basic', 'search_advanced'])

In [62]:
# Checks
assert enabled_features == ["login", "search_basic", "export"]
assert search_features == ["search_basic", "search_advanced"]

# Problem 33 — Build a lazy strict version of `compress`

Standard `compress` silently stops at the shorter iterable.

Write `compress_strict(data, selectors)` that behaves like `compress`, but raises `ValueError` if the two iterables have different lengths.

Keep the implementation lazy as much as possible.

In [63]:
# Solution

_sentinel = object()

def compress_strict(data, selectors):
    data_iter = iter(data)
    selector_iter = iter(selectors)

    while True:
        item = next(data_iter, _sentinel)
        selector = next(selector_iter, _sentinel)

        if item is _sentinel and selector is _sentinel:
            return

        if item is _sentinel or selector is _sentinel:
            raise ValueError("data and selectors must have the same length")

        if selector:
            yield item


assert_iterable_equal(compress_strict(["a", "b", "c"], [1, 0, 1]), ["a", "c"])

try:
    list(compress_strict(["a", "b", "c"], [1, 0]))
except ValueError as ex:
    strict_error = str(ex)

strict_error

'data and selectors must have the same length'

In [64]:
# Checks
assert strict_error == "data and selectors must have the same length"

# Problem 34 — Lazily select rows by column masks

Given table rows and a list of column selectors, return rows containing only selected columns.

Use `compress` inside a generator.

In [65]:
# Solution

headers = ["id", "name", "team", "score", "active"]
column_mask = [True, True, False, True, False]

rows = [
    [1, "Ada", "red", 91, True],
    [2, "Ben", "blue", 88, False],
    [3, "Cal", "red", 72, True],
]

selected_headers = list(compress(headers, column_mask))

def select_columns(rows, selectors):
    selectors = tuple(selectors)  # safe to reuse for every row
    for row in rows:
        yield list(compress(row, selectors))


selected_rows = list(select_columns(rows, column_mask))

selected_headers, selected_rows

(['id', 'name', 'score'], [[1, 'Ada', 91], [2, 'Ben', 88], [3, 'Cal', 72]])

In [66]:
# Checks
assert selected_headers == ["id", "name", "score"]
assert selected_rows == [[1, "Ada", 91], [2, "Ben", 88], [3, "Cal", 72]]

# Problem 35 — Build a reusable streaming query function

Write `query_records(records, *, where=None, reject=None, drop_until=None, take_until=None, mask=None)`.

Behavior:

- `drop_until`: drop records until this predicate is true; include the first matching record
- `take_until`: yield records until this predicate is true; do not include the first matching record
- `where`: keep records where predicate is true
- `reject`: reject records where predicate is true
- `mask`: selectors passed to `compress` after all previous filters

Return a lazy iterator.

In [67]:
# Solution

def query_records(records, *, where=None, reject=None, drop_until=None, take_until=None, mask=None):
    result = iter(records)

    if drop_until is not None:
        result = dropwhile(lambda record: not drop_until(record), result)

    if take_until is not None:
        result = takewhile(lambda record: not take_until(record), result)

    if where is not None:
        result = filter(where, result)

    if reject is not None:
        result = filterfalse(reject, result)

    if mask is not None:
        result = compress(result, mask)

    return result


records = [
    {"t": 1, "kind": "meta", "user": None, "amount": 0},
    {"t": 2, "kind": "start", "user": None, "amount": 0},
    {"t": 3, "kind": "event", "user": "ada", "amount": 100},
    {"t": 4, "kind": "event", "user": "ben", "amount": -5},
    {"t": 5, "kind": "event", "user": "cal", "amount": 250},
    {"t": 6, "kind": "event", "user": "dee", "amount": 40},
    {"t": 7, "kind": "end", "user": None, "amount": 0},
    {"t": 8, "kind": "event", "user": "eli", "amount": 999},
]

result = query_records(
    records,
    drop_until=lambda r: r["kind"] == "start",
    take_until=lambda r: r["kind"] == "end",
    where=lambda r: r["kind"] == "event",
    reject=lambda r: r["amount"] < 0,
    mask=[True, False, True],
)

queried = list(result)
queried

[{'t': 3, 'kind': 'event', 'user': 'ada', 'amount': 100},
 {'t': 6, 'kind': 'event', 'user': 'dee', 'amount': 40}]

In [68]:
# Checks
assert queried == [
    {"t": 3, "kind": "event", "user": "ada", "amount": 100},
    {"t": 6, "kind": "event", "user": "dee", "amount": 40},
]

# Problem 36 — Diagnose a subtle mask bug

A user wants to select the first and third original records. They write:

```python
compress(filter(is_approved, transactions), [True, False, True, False, False, False])
```

Explain why this does not select the first and third original records. Then fix it.

In [69]:
# Solution

transactions = [
    {"id": 1, "status": "approved"},
    {"id": 2, "status": "rejected"},
    {"id": 3, "status": "approved"},
    {"id": 4, "status": "approved"},
    {"id": 5, "status": "rejected"},
    {"id": 6, "status": "approved"},
]

def is_approved(tx):
    return tx["status"] == "approved"

buggy = list(compress(filter(is_approved, transactions), [True, False, True, False, False, False]))

# Correct approach when the mask refers to original positions:
original_position_mask = [True, False, True, False, False, False]
masked_originals = compress(transactions, original_position_mask)
fixed = list(filter(is_approved, masked_originals))

buggy, fixed

([{'id': 1, 'status': 'approved'}, {'id': 4, 'status': 'approved'}],
 [{'id': 1, 'status': 'approved'}, {'id': 3, 'status': 'approved'}])

In [70]:
# Checks
assert [tx["id"] for tx in buggy] == [1, 4]
assert [tx["id"] for tx in fixed] == [1, 3]

The bug happens because `compress` sees only the records that remain after filtering. Therefore the selector positions refer to approved records, not original records.

When your mask refers to original positions, apply `compress` before filtering.

# Problem 37 — Create a mini ETL stream

Build a lazy ETL pipeline for incoming dictionaries:

1. stop at `{"type": "shutdown"}`
2. keep only dictionaries with `type == "metric"`
3. keep only metrics where `name` is in an allowed set
4. reject metrics with `value is None`
5. transform kept records into `(name, value)` tuples

Then return a list.

In [71]:
# Solution

incoming = iter([
    {"type": "meta", "name": None, "value": None},
    {"type": "metric", "name": "cpu", "value": 0.4},
    {"type": "metric", "name": "debug", "value": 999},
    {"type": "metric", "name": "mem", "value": None},
    {"type": "metric", "name": "mem", "value": 0.7},
    {"type": "shutdown"},
    {"type": "metric", "name": "cpu", "value": 0.9},
])

allowed = {"cpu", "mem"}

pipeline = takewhile(lambda r: r.get("type") != "shutdown", incoming)
pipeline = filter(lambda r: r.get("type") == "metric", pipeline)
pipeline = filter(lambda r: r.get("name") in allowed, pipeline)
pipeline = filterfalse(lambda r: r.get("value") is None, pipeline)
pipeline = map(lambda r: (r["name"], r["value"]), pipeline)

metrics = list(pipeline)
metrics

[('cpu', 0.4), ('mem', 0.7)]

In [72]:
# Checks
assert metrics == [("cpu", 0.4), ("mem", 0.7)]

# Problem 38 — Design problem: choose the right tool

For each task, choose `filter`, `filterfalse`, `takewhile`, `dropwhile`, or `compress`.

1. Keep all positive numbers from an unsorted stream.
2. Skip CSV header lines until the first line that starts with `"id,"`.
3. Keep values until the first `None`.
4. Keep data items using a boolean mask from another stream.
5. Keep all records that do not have `status == "bad"`.

In [73]:
# Solution

answers = {
    1: "filter",
    2: "dropwhile",
    3: "takewhile",
    4: "compress",
    5: "filterfalse",
}

answers

{1: 'filter', 2: 'dropwhile', 3: 'takewhile', 4: 'compress', 5: 'filterfalse'}

In [74]:
# Checks
assert answers == {
    1: "filter",
    2: "dropwhile",
    3: "takewhile",
    4: "compress",
    5: "filterfalse",
}

# Problem 39 — Build a lazy selector from two predicates

Write `compress_by(data, selector_source, selector_predicate)`.

It should:

- convert `selector_source` into selectors by applying `selector_predicate`
- use those selectors to compress `data`
- remain lazy

Example: keep words whose matching score is at least `80`.

In [75]:
# Solution

def compress_by(data, selector_source, selector_predicate):
    selectors = map(selector_predicate, selector_source)
    return compress(data, selectors)


words = ["alpha", "beta", "gamma", "delta", "epsilon"]
scores = [90, 40, 81, 79, 100]

kept_words = list(compress_by(words, scores, lambda score: score >= 80))
kept_words

['alpha', 'gamma', 'epsilon']

In [76]:
# Checks
assert kept_words == ["alpha", "gamma", "epsilon"]

# Problem 40 — Final challenge: streaming fraud-review pipeline

You receive events sorted by time. Build a lazy pipeline that:

1. drops everything before the first `"OPEN_BATCH"` event
2. excludes the `"OPEN_BATCH"` marker
3. stops before `"CLOSE_BATCH"`
4. keeps only `"payment"` events
5. rejects events marked as `"test": True`
6. keeps payments with `amount >= 500`
7. applies an external review mask to the remaining payments
8. returns `(id, amount)` tuples

Use `dropwhile`, `takewhile`, `filter`, `filterfalse`, `compress`, and `map`.

In [77]:
# Solution

events = iter([
    {"type": "noise", "id": None, "amount": 0, "test": False},
    {"type": "OPEN_BATCH", "id": None, "amount": 0, "test": False},
    {"type": "payment", "id": "p1", "amount": 100, "test": False},
    {"type": "payment", "id": "p2", "amount": 700, "test": True},
    {"type": "payment", "id": "p3", "amount": 900, "test": False},
    {"type": "refund", "id": "r1", "amount": 800, "test": False},
    {"type": "payment", "id": "p4", "amount": 550, "test": False},
    {"type": "payment", "id": "p5", "amount": 1200, "test": False},
    {"type": "CLOSE_BATCH", "id": None, "amount": 0, "test": False},
    {"type": "payment", "id": "p6", "amount": 9999, "test": False},
])

after_open = dropwhile(lambda e: e["type"] != "OPEN_BATCH", events)
next(after_open)  # exclude OPEN_BATCH

inside_batch = takewhile(lambda e: e["type"] != "CLOSE_BATCH", after_open)
payments = filter(lambda e: e["type"] == "payment", inside_batch)
real_payments = filterfalse(lambda e: e["test"], payments)
large_payments = filter(lambda e: e["amount"] >= 500, real_payments)

review_mask = [True, False, True]
reviewed = compress(large_payments, review_mask)

result = list(map(lambda e: (e["id"], e["amount"]), reviewed))
result

[('p3', 900), ('p5', 1200)]

In [78]:
# Checks
assert result == [("p3", 900), ("p5", 1200)]

---

# Summary

You practiced advanced iterator selection patterns:

- `filter(predicate, iterable)` keeps values where the predicate is truthy.
- `filter(None, iterable)` keeps truthy items directly.
- `filterfalse(predicate, iterable)` keeps values where the predicate is falsy.
- `takewhile(predicate, iterable)` keeps an initial prefix and stops at the first failure.
- `dropwhile(predicate, iterable)` removes an initial prefix and then yields the rest.
- `compress(data, selectors)` keeps data values whose corresponding selectors are truthy.
- All of these tools are lazy iterators.
- Laziness saves memory, but skipped values are still consumed from upstream sources.
- Tool choice depends on whether you need global filtering, prefix/suffix logic, or mask-based selection.